# cAST-Scope — run via l'API d'inférence Hugging Face (SANS GPU/CUDA)

Contourne complètement l'instabilité GPU/CUDA de Colab rencontrée avec `colab_benchmark.ipynb` (redémarrages de kernel en boucle) : la génération se fait via un appel réseau à l'API d'inférence Hugging Face (`huggingface_hub.InferenceClient`), pas de modèle chargé localement, pas de GPU, pas de torch, pas de CUDA.

**Runtime** : `Runtime > Change runtime type > CPU` suffit (pas de GPU du tout requis) — plus rapide à allouer, gratuit, aucun risque de plantage lié au GPU.

**Limitation à connaître** : tous les modèles ne sont pas forcément hébergés sur l'API serverless gratuite de HF (certains gros modèles communautaires nécessitent un Inference Endpoint payant). Si un modèle renvoie une erreur "non disponible"/404, il faut soit un autre modèle servi par l'API, soit revenir à `colab_benchmark.ipynb` (GPU local).

## 1. Cloner (ou mettre à jour) le dépôt + installer les dépendances

Pas de torch/accelerate ici — seulement `huggingface_hub` (déjà une dépendance de `transformers`, mais on ne charge aucun modèle localement).

In [ ]:
import os

if os.path.isdir('/content/cAST-state'):
    %cd /content/cAST-state
    !git pull
else:
    !git clone --depth 1 https://github.com/Robertkiza0/cAST-state.git /content/cAST-state
    %cd /content/cAST-state

!pip install -q -r requirements.txt
!pip install -q huggingface_hub

print()
!echo "=== Commit actif : $(git rev-parse --short HEAD) — $(git log -1 --format=%s) ==="


## 2. Token Hugging Face (OBLIGATOIRE ici, pas optionnel)

L'API d'inférence a besoin d'un token pour l'authentification (voir `huggingface.co/settings/tokens` — créez-en un si besoin, secret Colab nommé exactement `HF_TOKEN`, icône clé 🔑 dans la barre latérale).

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN chargé.")


## 3. Télécharger les données RepoEval (tâches + 8 dépôts réels)

Idempotent : si déjà présent, ne retélécharge rien.

In [ ]:
import os
import zipfile

if os.path.isdir('data/repos_source') and os.listdir('data/repos_source'):
    print('RepoEval déjà présent, rien à faire.')
else:
    !rm -rf codet_src
    !git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
    %cd codet_src
    !git sparse-checkout init --cone
    !git sparse-checkout set RepoCoder
    !git checkout main
    %cd ..

    with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
        z.extractall('datasets rapo')
    with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
        z.extractall('data/repos_source')

    print('RepoEval : dataset et dépôts extraits.')


## 4. Test minimal du générateur (isolé, appel réseau seul, pas de harnais complet)

In [ ]:
import time
from generation import HFInferenceGenerator

MODEL_NAME = 'bigcode/starcoder2-7b'  # changer ici si ce modèle n'est pas dispo sur l'API serverless

gen = HFInferenceGenerator(MODEL_NAME)
t0 = time.time()
try:
    output = gen.generate("def add(a, b):\n    return a + b\n\ndef multiply(a, b):\n    return ")
    print(f'generate() en {time.time()-t0:.1f}s')
    print(f'Sortie: {output!r}')
except RuntimeError as e:
    print(f'Échec — ce modèle n\'est probablement pas servi par l\'API gratuite: {e}')


## 5. Test rapide, générateur factice (valide tout le pipeline sans appel réseau)

In [ ]:
%run -i /content/cAST-state/run_benchmark.py --dataset repoeval --n-tasks 10 --generator stub


## 6. Run réel via l'API d'inférence — petit d'abord

`--n-tasks 20` pour un premier passage. Chaque tâche fait 3 appels réseau (un par stratégie) — plus lent qu'un GPU local par tâche, mais aucun risque de plantage de kernel.

In [ ]:
%run -i /content/cAST-state/run_benchmark.py --dataset repoeval --n-tasks 20 --generator hf_api \
    --model-name bigcode/starcoder2-7b


## 7. Run complet — RepoEval + CrossCodeEval

`--tasks-per-repo 38` pour approcher 300 tâches sur les 8 dépôts RepoEval.

In [ ]:
%run -i /content/cAST-state/run_benchmark.py --dataset both --n-tasks 300 --tasks-per-repo 38 \
    --generator hf_api --model-name bigcode/starcoder2-7b


## 8. (Optionnel) Run complet — CodeLlama-7B-Python

In [ ]:
%run -i /content/cAST-state/run_benchmark.py --dataset both --n-tasks 300 --tasks-per-repo 38 \
    --generator hf_api --model-name codellama/CodeLlama-7b-Python-hf


## Télécharger les résultats (le runtime Colab est éphémère)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/cast_state_results', 'zip', '/content/cAST-state/results')
files.download('/content/cast_state_results.zip')


## Notes

- Pass@1 == Exact Match ici (pas de harnais d'exécution — voir `metrics.py:compute_pass_at_1`).
- Chaque appel réseau ajoute de la latence réseau (souvent plus lent qu'un GPU local par tâche individuelle) mais élimine tout risque de plantage GPU/CUDA/kernel — un compromis débit contre stabilité.
- Si un modèle n'est pas disponible sur l'API serverless (erreur 404/501), essayez un autre modèle plus couramment servi, ou repassez sur `colab_benchmark.ipynb` (GPU local) pour ce modèle précis.
- Toutes les cellules d'installation sont idempotentes — sûres à relancer après un redémarrage.